# Fraud Detection — Random Forest

This notebook extends the fraud detection analysis by evaluating a Random Forest classifier as an alternative to the Logistic Regression baseline.

The Logistic Regression model developed in the previous notebook showed poor fraud detection performance despite feature scaling and classification threshold analysis. This suggests that the problem cannot be resolved simply by improving model convergence or changing the decision threshold.

Random Forest is introduced as a more flexible, non-linear model capable of capturing interactions and patterns that Logistic Regression may not represent effectively.

The main goals are to:
- prepare the fraud dataset for Random Forest modeling,
- train and evaluate a Random Forest classifier,
- analyze its fraud detection performance using recall, precision and F1-score,
- compare its performance with the previous Logistic Regression baseline,
- determine whether a more flexible model provides better separation between fraudulent and non-fraudulent transactions.

In [1]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
)

from finsight.database import (
    connect_to_database,
    test_database_connection,
)

from finsight.fraud_data import (
    download_train_data,
    download_validation_data,
)

from finsight.fraud_preprocessing import (
    encode_categorical_features,
    handle_missing_values,
)

from finsight.validation import (
    check_missing_values
)

In [2]:
engine = connect_to_database()
test_database_connection(engine)

Connected!


In [3]:
X_train, y_train = download_train_data(engine)
X_val, y_val = download_validation_data(engine)

In [4]:
final_train, final_val = encode_categorical_features(X_train, X_val)

In [5]:
check_missing_values(final_train)

amount                              0
mcc_key                             0
merchant_id                         0
transaction_hour                    0
day_of_week                         0
is_weekend                          0
user_previous_transaction_count     0
user_previous_avg_amount           30
amount_vs_user_avg                 30
card_previous_transaction_count     0
card_previous_avg_amount           93
amount_vs_card_avg                 93
use_chip_Chip Transaction           0
use_chip_Online Transaction         0
use_chip_Swipe Transaction          0
dtype: int64

In [6]:
check_missing_values(final_val)

amount                              0
mcc_key                             0
merchant_id                         0
transaction_hour                    0
day_of_week                         0
is_weekend                          0
user_previous_transaction_count     0
user_previous_avg_amount            0
amount_vs_user_avg                  0
card_previous_transaction_count     0
card_previous_avg_amount           45
amount_vs_card_avg                 45
use_chip_Chip Transaction           0
use_chip_Online Transaction         0
use_chip_Swipe Transaction          0
dtype: int64

In [7]:
final_train, final_val = handle_missing_values(final_train, final_val)

In [8]:
check_missing_values(final_train)

amount                             0
mcc_key                            0
merchant_id                        0
transaction_hour                   0
day_of_week                        0
is_weekend                         0
user_previous_transaction_count    0
user_previous_avg_amount           0
amount_vs_user_avg                 0
card_previous_transaction_count    0
card_previous_avg_amount           0
amount_vs_card_avg                 0
use_chip_Chip Transaction          0
use_chip_Online Transaction        0
use_chip_Swipe Transaction         0
dtype: int64

In [9]:
check_missing_values(final_val)

amount                             0
mcc_key                            0
merchant_id                        0
transaction_hour                   0
day_of_week                        0
is_weekend                         0
user_previous_transaction_count    0
user_previous_avg_amount           0
amount_vs_user_avg                 0
card_previous_transaction_count    0
card_previous_avg_amount           0
amount_vs_card_avg                 0
use_chip_Chip Transaction          0
use_chip_Online Transaction        0
use_chip_Swipe Transaction         0
dtype: int64

## Baseline Random Forest Model

In [10]:
rf = RandomForestClassifier(random_state=42)
rf.fit(final_train, y_train)
print("Training set score: {:.4f}".format(rf.score(final_train, y_train)))
print("Validation set score: {:.4f}".format(rf.score(final_val, y_val)))

Training set score: 1.0000
Validation set score: 0.9974


In [11]:
y_pred = rf.predict(final_val)
y_pred.sum()

np.int64(909)

### Recall

In [12]:
recall = recall_score(y_val, y_pred)
print("Recall: {:.4f}".format(recall))

Recall: 0.0411


### Precision

In [13]:
precision = precision_score(y_val, y_pred)
print("Precision: {:.4f}".format(precision))

Precision: 0.0737


### F1

In [14]:
f1 = f1_score(y_val, y_pred)
print("F1: {:.4f}".format(f1))

F1: 0.0528


### Confusion Matrix

In [15]:
conf_matrix = confusion_matrix(y_val, y_pred)
print(conf_matrix)

[[932128    842]
 [  1562     67]]


### Controlling Tree Depth

The baseline Random Forest achieves almost perfect accuracy on the training data, while its fraud detection performance on the validation set remains poor. This suggests that the model may be overfitting the training data.

To reduce model complexity, the maximum depth of individual trees will be limited using the `max_depth` parameter.

The goal is to investigate whether shallower trees can generalize better to unseen transactions and improve fraud detection performance.

## Random Forest with Limited Tree Depth

In [16]:
rf_limited10 = RandomForestClassifier(max_depth=10, random_state=42)
rf_limited10.fit(final_train, y_train)
print("Training set score: {:.4f}".format(rf_limited10.score(final_train, y_train)))
print("Validation set score: {:.4f}".format(rf_limited10.score(final_val, y_val)))

Training set score: 0.9835
Validation set score: 0.9975


In [17]:
y_pred_limited = rf_limited10.predict(final_val)

In [18]:
recall_limited10 = recall_score(y_val, y_pred_limited)
precision_limited10 = precision_score(y_val, y_pred_limited)
f1_limited10 = f1_score(y_val, y_pred_limited)
print("Recall: {:.4f}".format(recall_limited10))
print("Precision: {:.4f}".format(precision_limited10))
print("F1: {:.4f}".format(f1_limited10))

Recall: 0.0264
Precision: 0.0522
F1: 0.0351


In [19]:
rf_limited20 = RandomForestClassifier(max_depth=20, random_state=42)
rf_limited20.fit(final_train, y_train)
print("Training set score: {:.4f}".format(rf_limited20.score(final_train, y_train)))
print("Validation set score: {:.4f}".format(rf_limited20.score(final_val, y_val)))

Training set score: 0.9964
Validation set score: 0.9974


In [20]:
y_pred_limited20 = rf_limited20.predict(final_val)

In [21]:
recall_limited20 = recall_score(y_val, y_pred_limited20)
precision_limited20 = precision_score(y_val, y_pred_limited20)
f1_limited20 = f1_score(y_val, y_pred_limited20)
print("Recall: {:.4f}".format(recall_limited20))
print("Precision: {:.4f}".format(precision_limited20))
print("F1: {:.4f}".format(f1_limited20))

Recall: 0.0387
Precision: 0.0685
F1: 0.0494


In [23]:
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "max_depth=10",
        "max_depth=20"
    ],
    "Recall": [
        recall,
        recall_limited10,
        recall_limited20
    ],
    "Precision": [
        precision,
        precision_limited10,
        precision_limited20
    ],
    "F1": [
        f1,
        f1_limited10,
        f1_limited20
    ]
})

results = pd.DataFrame(results).set_index("Model")
results

,Recall,Precision,F1
Model,,,
Baseline,0.041130,0.073707,0.052797
max_depth=10,0.026397,0.052184,0.035059
max_depth=20,0.038674,0.068478,0.049431


### Comparison Summary

Limiting the maximum tree depth reduced the model's ability to fit the training data, but did not improve fraud detection on the validation set.

- `max_depth=10` produced the weakest recall, precision and F1-score.
- `max_depth=20` performed better than `max_depth=10`, but still did not outperform the baseline Random Forest.
- The baseline model without a depth limit achieved the highest recall, precision and F1-score.

These results suggest that limiting tree depth alone does not improve the model's ability to detect fraudulent transactions. The next experiment will therefore focus on the class imbalance problem.